In [ ]:
# OpenBMI (Lee2019_MI) cache for the OpenBMI LOSO runs. Internet on; no GPU needed.
# Downloads each subject (~1.1 GB of MAT files), preprocesses it (4-38 Hz, CAR,
# 100 Hz, 0-4 s), writes openbmi_SXX.npz (~20 MB) and deletes the raw files.
# Afterwards: Output -> New Dataset, then Add Input in the three training
# notebooks. If the session ends early, attach this notebook's output as input
# and rerun: cached subjects are copied over and only the rest is downloaded.
import codecs
import os
import re
import select
import shutil
import subprocess
import sys
import time
from pathlib import Path

# ---------------- Job configuration ----------------
SUBJECTS = '1-54'

BRANCH = 'feature/hada-autodial-openbmi'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
REPO_PATH = Path("/kaggle/working/tcformer-autodial-openbmi-cache")
MNE_DATA = Path("/tmp/mne_data")   # raw MAT files, outside the saved output
CACHE_DIR = Path("/kaggle/working/openbmi_cache")
BUILD_LOG = Path("/kaggle/working/openbmi_cache_build.log")


def run(command, cwd=None, env=None, stream=False, log_path=None):
    command = list(map(str, command))
    print("+", " ".join(command), flush=True)
    if not stream:
        subprocess.run(
            command,
            cwd=str(cwd) if cwd else None,
            env=env,
            check=True,
        )
        return

    process = subprocess.Popen(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    started_at = time.monotonic()
    with open(log_path, "w", encoding="utf-8") as log_file:
        while True:
            ready, _, _ = select.select([process.stdout], [], [], 30)
            if ready:
                chunk = os.read(process.stdout.fileno(), 4096)
                if not chunk:
                    break
                output = decoder.decode(chunk)
                print(output, end="", flush=True)
                log_file.write(output)
                log_file.flush()
            elif process.poll() is None:
                elapsed = (time.monotonic() - started_at) / 60
                heartbeat = f"[Kaggle heartbeat] running | elapsed={elapsed:.1f}m\n"
                print(heartbeat, end="", flush=True)
                log_file.write(heartbeat)
                log_file.flush()
            else:
                break

        remaining = decoder.decode(b"", final=True)
        if remaining:
            print(remaining, end="", flush=True)
            log_file.write(remaining)

    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f"Process exited with code {return_code}. Full log: {log_path}"
        )


# ---------------- Clone branch and verify the implementation ----------------
run([sys.executable, "-m", "pip", "install", "-q", "uv"])
if (REPO_PATH / ".git").is_dir():
    run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_PATH)
    run(["git", "fetch", "origin", BRANCH], cwd=REPO_PATH)
    # A previous run edits the YAML locally; reset so reruns are safe.
    run(["git", "checkout", "--force", BRANCH], cwd=REPO_PATH)
    run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_PATH)
else:
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_PATH])
run(["git", "log", "-1", "--oneline"], cwd=REPO_PATH)

pipeline_text = (REPO_PATH / "train_pipeline.py").read_text(encoding="utf-8")
model_text = (REPO_PATH / "models/hada_tcformer.py").read_text(encoding="utf-8")
datamodule_text = (REPO_PATH / "utils/get_datamodule_cls.py").read_text(encoding="utf-8")
assert "ModelCheckpoint" in pipeline_text and 'monitor="val_acc"' in pipeline_text
assert "class AutoDIALFeatureNorm" in model_text
assert "openbmi_loso" in datamodule_text
assert (REPO_PATH / "build_openbmi_cache.py").is_file()
print("Verified: OpenBMI cache builder.")

UV = shutil.which("uv") or "uv"
run([UV, "venv", "--clear", "--python", "3.10", ".venv"], cwd=REPO_PATH)
PYTHON = REPO_PATH / ".venv/bin/python"
run([UV, "pip", "install", "--python", PYTHON, "torch==2.7.1", "torchvision==0.22.1",
     "--index-url", "https://download.pytorch.org/whl/cu126"])
run([UV, "pip", "install", "--python", PYTHON, "-r", "requirements.txt"], cwd=REPO_PATH)

# ---------------- Build the cache ----------------
MNE_DATA.mkdir(parents=True, exist_ok=True)
environment = os.environ.copy()
environment.update({
    "PYTHONUNBUFFERED": "1",
    "MPLBACKEND": "Agg",
    "MNE_DATA": str(MNE_DATA),
    "MNE_DATASETS_LEE2019_MI_PATH": str(MNE_DATA),
    "OPENBMI_CACHE_INPUT": "/kaggle/input",
})
run([PYTHON, "-u", "build_openbmi_cache.py", "--subjects", SUBJECTS, "--out", CACHE_DIR],
    cwd=REPO_PATH, env=environment, stream=True, log_path=BUILD_LOG)
# Keep only the cache in the saved output.
shutil.rmtree(REPO_PATH, ignore_errors=True)
print("Cache files:", len(list(CACHE_DIR.glob("openbmi_S*.npz"))), "in", CACHE_DIR)
